
# Phase 1 – Network Audit & ITD Federation Check

This notebook performs:

1. Synoptic network audit (networks present in Idaho)
2. Metadata completeness baseline by network
3. Clean visual outputs for presentation

Outputs saved in `out/`


In [ ]:

import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)

SYNOPTIC_TOKEN = os.getenv("SYNOPTIC_TOKEN", "").strip()
STATIONS_ALL_CSV = os.path.join(OUT_DIR, "stations_all.csv")

assert os.path.exists(STATIONS_ALL_CSV), "stations_all.csv not found. Run Phase 1 Notebook 1 first."
stations_all = pd.read_csv(STATIONS_ALL_CSV)

print("Stations loaded:", len(stations_all))


In [ ]:

# -----------------------------
# 1) Fetch Synoptic networks
# -----------------------------

def fetch_synoptic_networks(token):
    url = "https://api.synopticdata.com/v2/networks"
    params = {"token": token}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    js = r.json()
    df = pd.DataFrame(js.get("MNET", []))
    
    if "ID" in df.columns:
        df["mnet_id"] = df["ID"].astype(str).str.strip()
    elif "id" in df.columns:
        df["mnet_id"] = df["id"].astype(str).str.strip()
    else:
        raise ValueError("No network ID column found.")
        
    return df

networks_df = fetch_synoptic_networks(SYNOPTIC_TOKEN)
print("Networks in Synoptic:", len(networks_df))


In [ ]:

# -----------------------------
# 2) Networks present in Idaho
# -----------------------------

syn = stations_all[stations_all.source == "SYNOPTIC"].copy()
syn["network_id"] = syn["network_id"].astype(str).str.strip()

counts = syn.groupby("network_id").size().reset_index(name="n_stations")
counts.rename(columns={"network_id": "mnet_id"}, inplace=True)

networks_in_idaho = counts.merge(networks_df, on="mnet_id", how="left")
networks_in_idaho.sort_values("n_stations", ascending=False, inplace=True)

networks_in_idaho.to_csv(os.path.join(OUT_DIR, "networks_in_idaho.csv"), index=False)
print("Networks in Idaho:", len(networks_in_idaho))
networks_in_idaho.head()


In [ ]:

# -----------------------------
# 3) Metadata completeness baseline
# -----------------------------

def metadata_completeness(df):
    rows = []
    for mnet_id, g in df.groupby("network_id"):
        rows.append({
            "mnet_id": mnet_id,
            "n_stations": len(g),
            "has_latlon_pct": 100.0 * g[["latitude","longitude"]].notna().all(axis=1).mean(),
            "has_elevation_pct": 100.0 * g["elevation_m"].notna().mean(),
        })
    return pd.DataFrame(rows).sort_values("n_stations", ascending=False)

meta_comp = metadata_completeness(syn)
meta_comp.to_csv(os.path.join(OUT_DIR, "metadata_completeness_by_network.csv"), index=False)

meta_comp.head()


In [ ]:

# -----------------------------
# 4) Plot: Top networks
# -----------------------------

top = networks_in_idaho.head(10)

plt.figure(figsize=(8,5))
plt.barh(top["mnet_id"], top["n_stations"])
plt.xlabel("Number of stations in Idaho")
plt.title("Top Synoptic Networks in Idaho")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_top_networks.png"), dpi=150)
plt.show()
